# Explore GHTorrent Tables for Sentiment Mapping (Notebook 2)

This notebook helps you understand where sentiment-labeled comments are stored in GHTorrent and how they connect to projects. These checks are for exploration and validation. You do not need to run every query to run the end-to-end workflow.

### Planned Output
By the end of this notebook, you should have:
1. A clear view of how sentiment comments are split across commit vs PR comment tables
2. A ranked list of projects with sentiment-labeled commit comments
3. A ranked list of projects with sentiment-labeled PR comments
4. A global summary of comments reachable from canonical repos vs forks

### Check 1: How sentiment comments are distributed

Use these queries to see how many sentiment comments are in commit comments, PR comments, and both tables.

Count sentiment comments in `commit_comments` (expected: 4317):

```sql
SELECT COUNT(*)
FROM comment_sentiment s
INNER JOIN commit_comments cc ON s.ID = cc.comment_id;
```

---

Count sentiment comments in `pull_request_comments` (expected: 2890):

```sql
SELECT COUNT(*)
FROM comment_sentiment s
INNER JOIN pull_request_comments prc ON s.ID = prc.comment_id;
```

---

Count overlap that appears in both tables (expected: 85):

```sql
SELECT COUNT(*) AS both_tables
FROM comment_sentiment s
INNER JOIN commit_comments cc ON s.ID = cc.comment_id
INNER JOIN pull_request_comments prc ON s.ID = prc.comment_id;
```

Quick interpretation:
- Commit-only = 4317 - 85 = 4232
- PR-only = 2890 - 85 = 2805
- Both = 85
- Total unique comments = 7122

### Check 2: Projects with the most sentiment-labeled commit comments

Use this to rank projects by number of labeled commit comments.

```sql
SELECT p.id, p.name, p.url, COUNT(DISTINCT s.ID) AS labeled_comment_count
FROM projects p
INNER JOIN commits c ON p.id = c.project_id
INNER JOIN commit_comments cc ON c.id = cc.commit_id
INNER JOIN comment_sentiment s ON cc.comment_id = s.ID
GROUP BY p.id, p.name, p.url
ORDER BY labeled_comment_count DESC;
```

---

Use this to inspect example rows for one project (replace `{owner}` and `{repo}`):

```sql
SELECT c.sha, p.url, p.name, s.ID AS comment_id, s.Text AS comment_text
FROM commits c
INNER JOIN projects p ON c.project_id = p.id
INNER JOIN commit_comments cc ON c.id = cc.commit_id
INNER JOIN comment_sentiment s ON cc.comment_id = s.ID
WHERE p.url = 'https://api.github.com/repos/{owner}/{repo}';
```

### Check 3: Projects with the most sentiment-labeled PR comments

Use this to rank projects by number of labeled PR comments.

```sql
SELECT p.id, p.name, p.url, COUNT(DISTINCT s.ID) AS labeled_comment_count
FROM projects p
INNER JOIN pull_requests pr ON p.id = pr.base_repo_id
INNER JOIN pull_request_comments prc ON pr.id = prc.pull_request_id
INNER JOIN comment_sentiment s ON prc.comment_id = s.ID
GROUP BY p.id, p.name, p.url
ORDER BY labeled_comment_count DESC;
```

### Check 4: Canonical repo vs fork accessibility summary

This query estimates how many sentiment comments are reachable from canonical repos vs only from forks.

```sql
WITH RECURSIVE project_root AS (
  SELECT p.id AS project_id, p.id AS root_id
  FROM projects p
  WHERE p.forked_from IS NULL
  UNION ALL
  SELECT c.id AS project_id, pr.root_id
  FROM projects c
  JOIN project_root pr ON c.forked_from = pr.project_id
),
comment_project_rows AS (
  SELECT cs.ID AS comment_id, c.project_id, 'commit_comment' AS source_tag
  FROM comment_sentiment cs
  JOIN commit_comments cc ON cs.ID = cc.comment_id
  JOIN commits c ON cc.commit_id = c.id

  UNION ALL

  SELECT cs.ID AS comment_id, pr.base_repo_id AS project_id, 'pr_comment' AS source_tag
  FROM comment_sentiment cs
  JOIN pull_request_comments prc ON cs.ID = prc.comment_id
  JOIN pull_requests pr ON prc.pull_request_id = pr.id

  UNION ALL

  SELECT cs.ID AS comment_id, pr.head_repo_id AS project_id, 'pr_comment' AS source_tag
  FROM comment_sentiment cs
  JOIN pull_request_comments prc ON cs.ID = prc.comment_id
  JOIN pull_requests pr ON prc.pull_request_id = pr.id
),
labeled AS (
  SELECT
    cpr.comment_id,
    cpr.source_tag,
    pr.root_id,
    (cpr.project_id = pr.root_id) AS is_canonical
  FROM comment_project_rows cpr
  JOIN project_root pr ON pr.project_id = cpr.project_id
),
comment_flags AS (
  SELECT
    root_id,
    source_tag,
    comment_id,
    MAX(CASE WHEN is_canonical THEN 1 ELSE 0 END) AS has_canonical,
    MAX(CASE WHEN NOT is_canonical THEN 1 ELSE 0 END) AS has_fork
  FROM labeled
  GROUP BY root_id, source_tag, comment_id
),
global_counts AS (
  SELECT
    COUNT(*) AS mapped_comment_ids,
    SUM(CASE WHEN has_canonical = 1 THEN 1 ELSE 0 END) AS canonical_accessible,
    SUM(CASE WHEN has_fork = 1 THEN 1 ELSE 0 END) AS fork_accessible,
    SUM(CASE WHEN has_canonical = 1 AND has_fork = 0 THEN 1 ELSE 0 END) AS canonical_only,
    SUM(CASE WHEN has_canonical = 0 AND has_fork = 1 THEN 1 ELSE 0 END) AS fork_only,
    SUM(CASE WHEN has_canonical = 1 AND has_fork = 1 THEN 1 ELSE 0 END) AS both_sides
  FROM comment_flags
)
SELECT
  mapped_comment_ids,
  canonical_accessible,
  fork_accessible,
  canonical_only,
  fork_only,
  both_sides,
  ROUND(100 * fork_only / NULLIF(mapped_comment_ids, 0), 2) AS fork_only_pct,
  ROUND(100 * canonical_only / NULLIF(mapped_comment_ids, 0), 2) AS canonical_only_pct,
  ROUND(100 * (canonical_only + both_sides) / NULLIF(mapped_comment_ids, 0), 2) AS canonical_reachable_pct
FROM global_counts;
```

Expected values from prior runs:
- `canonical_only`: 4555
- `fork_only`: 569
- `both_sides`: 2083
- Canonical reachable rate: about 93.2%

### When to move on to Notebook 3

You can move to Notebook 3 when all of these are true:

1. Check 1 totals are consistent (commit + PR - overlap = 7122).
2. Check 2 returns project rows for commit-comment mappings (not empty).
3. Check 3 returns project rows for PR-comment mappings (not empty).
4. Check 4 runs successfully and shows non-zero canonical reachability.

If any check is empty or fails, fix the data/join issue first before moving on.